In [2]:
# Check Python and package versions before setting up model training.
# This does not install or change anything.

import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

for package in ["tensorflow", "keras", "tfswin", "numpy"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.11.17
tensorflow: 2.17.1
keras: 3.4.1
tfswin: 4.1.1
numpy: 1.26.4


In [3]:
# Verify that the notebook uses the training environment
# and that the model libraries import successfully.

import sys
import tensorflow as tf
import keras
import numpy as np
from tfswin import SwinTransformerV2Tiny256
from importlib.metadata import version

print("Python executable:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("NumPy:", np.__version__)
print("tfswin:", version("tfswin"))
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("Imports successful.")

Python executable: /Users/hf/Documents/CassaDx/.venv-train/bin/python
TensorFlow: 2.17.1
Keras: 3.4.1
NumPy: 1.26.4
tfswin: 4.1.1
Available GPUs: []
Imports successful.


In [4]:
# Load pretrained DenseNet-121, freeze its layers, and test
# whether it produces feature maps from a 256 × 256 input.

dense_branch = keras.applications.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
)

dense_branch.trainable = False

# A dummy image checks the model's operation, not its accuracy.
dummy_image = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
dense_input = keras.applications.densenet.preprocess_input(dummy_image)

dense_features = dense_branch(dense_input, training=False)

print("Feature-map shape:", dense_features.shape)
print("Trainable variables:", len(dense_branch.trainable_variables))
print("All values finite:",
      bool(tf.reduce_all(tf.math.is_finite(dense_features)).numpy()))

Feature-map shape: (1, 8, 8, 1024)
Trainable variables: 0
All values finite: True


In [5]:
# Load pretrained SwinV2-Tiny, freeze its layers, and test
# its feature output. Built-in preprocessing handles raw 0–255 pixels.

swin_branch = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)

swin_branch.trainable = False

dummy_image = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
swin_features = swin_branch(dummy_image, training=False)

print("Feature-map shape:", swin_features.shape)
print("Trainable variables:", len(swin_branch.trainable_variables))
print("All values finite:",
      bool(tf.reduce_all(tf.math.is_finite(swin_features)).numpy()))

Feature-map shape: (1, 8, 8, 768)
Trainable variables: 0
All values finite: True


In [6]:
# Combine the frozen DenseNet and SwinV2 branches.
# Train only the new five-class classifier in this initial baseline.

inputs = keras.Input(shape=(256, 256, 3), name="leaf_image")

# Each branch receives its required preprocessing exactly once.
dense_inputs = keras.applications.densenet.preprocess_input(inputs)
dense_maps = dense_branch(dense_inputs, training=False)
swin_maps = swin_branch(inputs, training=False)

dense_vector = keras.layers.GlobalAveragePooling2D(
    name="dense_pool"
)(dense_maps)

swin_vector = keras.layers.GlobalAveragePooling2D(
    name="swin_pool"
)(swin_maps)

combined = keras.layers.Concatenate(name="feature_fusion")(
    [dense_vector, swin_vector]
)

outputs = keras.layers.Dense(
    5, activation="softmax", name="disease_classifier"
)(combined)

hybrid_model = keras.Model(inputs, outputs, name="CassaDx_baseline")

# Check the combined model using one dummy image.
predictions = hybrid_model(dummy_image, training=False)

print("Output shape:", predictions.shape)
print("Probability sum:", float(tf.reduce_sum(predictions[0]).numpy()))
print("Trainable parameters:",
      sum(int(np.prod(v.shape)) for v in hybrid_model.trainable_variables))
print("All values finite:",
      bool(tf.reduce_all(tf.math.is_finite(predictions)).numpy()))

Output shape: (1, 5)
Probability sum: 1.0
Trainable parameters: 8965
All values finite: True


In [7]:
# Run one real training image through the hybrid to check the
# complete input path. This does not train or evaluate accuracy.

import csv
import json
from pathlib import Path
from PIL import Image

data_dir = Path("data")
split_dir = Path("splits")

with (split_dir / "train.csv").open(newline="") as file:
    train_records = list(csv.DictReader(file))

with (split_dir / "training_config.json").open() as file:
    training_config = json.load(file)

disease_names = training_config["label_to_disease"]
sample = train_records[0]

with Image.open(data_dir / "train_images" / sample["image_id"]) as img:
    img = img.convert("RGB").resize(
        (256, 256), Image.Resampling.BILINEAR
    )
    image_array = np.array(img, dtype=np.float32)

# Add the batch dimension. Keep raw pixels in the 0–255 range:
# the model handles normalization separately for each branch.
image_batch = image_array[np.newaxis, ...]
probabilities = hybrid_model(image_batch, training=False).numpy()[0]

print("Dataset label:", disease_names[sample["label"]])
print("Input shape:", image_batch.shape)
print("All outputs finite:", bool(np.isfinite(probabilities).all()))
print("Probability sum:", float(probabilities.sum()))

Dataset label: Healthy
Input shape: (1, 256, 256, 3)
All outputs finite: True
Probability sum: 1.0


In [8]:
# Export the current hybrid and attempt a standard FP32 TFLite conversion.
# This checks compatibility before we spend time training.

from pathlib import Path

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

saved_model_dir = export_dir / "hybrid_export_check"
tflite_path = export_dir / "hybrid_export_check_fp32.tflite"

hybrid_model.export(str(saved_model_dir))

converter = tf.lite.TFLiteConverter.from_saved_model(
    str(saved_model_dir)
)

# Require standard TFLite operations for this first compatibility check.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

tflite_model = converter.convert()
tflite_path.write_bytes(tflite_model)

print("\nFP32 conversion successful.")
print("Saved:", tflite_path)
print("File size (MiB):", round(len(tflite_model) / (1024 ** 2), 2))

INFO:tensorflow:Assets written to: models/hybrid_export_check/assets


INFO:tensorflow:Assets written to: models/hybrid_export_check/assets


Saved artifact at 'models/hybrid_export_check'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='leaf_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  5072211216: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5067255504: TensorSpec(shape=(3,), dtype=tf.float32, name=None)
  5067259728: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5067258576: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5067259536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067259152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067258960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067257040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067257808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5068481040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5068484112: TensorSpec(shape

W0000 00:00:1792154809.020823 11798197 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792154809.020860 11798197 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 17:46:49.021487: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_export_check
2026-10-16 17:46:49.041443: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 17:46:49.041460: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_export_check
2026-10-16 17:46:49.259007: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-10-16 17:46:49.293200: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 17:46:50.633555: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_export_check
2026-10-16 17:46:51.046438: I ten

ConverterError: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(callsite(fused["Roll:", "CassaDx_baseline_1/swin2_tiny_256_1/layers.0_1/blocks.0_1/Roll@__inference___call___49330"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___52657"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): 'tf.Roll' op is neither a custom op nor a flex op
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(callsite(fused["Roll:", "CassaDx_baseline_1/swin2_tiny_256_1/layers.0_1/blocks.0_1/Roll@__inference___call___49330"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___52657"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): Error code: ERROR_NEEDS_FLEX_OPS
<unknown>:0: error: loc(callsite(callsite(fused["Transpose:", "CassaDx_baseline_1/swin2_tiny_256_1/layers

In [9]:
# Test FP32 export with TensorFlow fallback operations enabled.
# This is a diagnostic artifact, not a full-INT8 deployment model.

converter = tf.lite.TFLiteConverter.from_saved_model(
    "models/hybrid_export_check"
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

fallback_model = converter.convert()

fallback_path = Path("models/hybrid_export_check_flex_fp32.tflite")
fallback_path.write_bytes(fallback_model)

print("\nFallback FP32 conversion successful.")
print("Saved:", fallback_path)
print("File size (MiB):",
      round(len(fallback_model) / (1024 ** 2), 2))

W0000 00:00:1792154824.588836 11798197 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792154824.588852 11798197 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 17:47:04.589076: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_export_check
2026-10-16 17:47:04.607467: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 17:47:04.607483: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_export_check
2026-10-16 17:47:04.845259: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 17:47:06.146814: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_export_check
2026-10-16 17:47:06.603248: I tensorflow/cc/saved_model/loader.cc:462] SavedModel load for tags { serve }; Status: success: OK. Took 2014172 microseconds.
2026-10-16 


Fallback FP32 conversion successful.
Saved: models/hybrid_export_check_flex_fp32.tflite
File size (MiB): 143.26


In [ ]:
# Compare the exported FP32 model with the original Keras model
# using the same real image. This checks numerical agreement, not accuracy.

interpreter = tf.lite.Interpreter(
    model_path="models/hybrid_export_check_flex_fp32.tflite"
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

test_input = image_batch.astype(np.float32)

keras_output = hybrid_model(test_input, training=False).numpy()

interpreter.set_tensor(input_info["index"], test_input)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_info["index"])

max_difference = np.max(np.abs(keras_output - tflite_output))

print("Keras probabilities:", keras_output[0])
print("TFLite probabilities:", tflite_output[0])
print("Maximum absolute difference:", float(max_difference))
print("Outputs close:",
      np.allclose(keras_output, tflite_output, atol=1e-4, rtol=1e-3))

: 

In [1]:
# Test equivalent window operations using transposes of at most 6 dimensions.
# This cell does not modify the installed library or build the model.

import tensorflow as tf
from keras import ops
from tfswin.window import (
    window_partition,
    window_reverse,
    window_partition_fused,
    window_reverse_fused,
)

def partition_for_export(
    inputs, height, width, window_size, num_heads,
    qkv_mult=3, dtype=None, name=None
):
    windows = window_partition(
        inputs, height, width, window_size, dtype=dtype, name=name
    )
    head_channels = windows.shape[-1] // (qkv_mult * num_heads)

    outputs = ops.reshape(
        windows,
        [-1, window_size ** 2, qkv_mult, num_heads, head_channels]
    )
    return ops.transpose(outputs, [2, 0, 3, 1, 4])


def reverse_for_export(
    inputs, height, width, window_size, num_heads,
    dtype=None, name=None
):
    head_channels = inputs.shape[-1]
    outputs = ops.transpose(inputs, [0, 2, 1, 3])
    outputs = ops.reshape(
        outputs,
        [-1, window_size ** 2, num_heads * head_channels]
    )
    return window_reverse(
        outputs, height, width, window_size, dtype=dtype, name=name
    )


# Compare the original and replacement operations on identical test data.
x = tf.reshape(
    tf.range(32 * 32 * 288, dtype=tf.float32),
    (1, 32, 32, 288)
)

original_partition = window_partition_fused(x, 32, 32, 16, 3)
replacement_partition = partition_for_export(x, 32, 32, 16, 3)
tf.debugging.assert_equal(original_partition, replacement_partition)

test_windows = original_partition[2]
original_reverse = window_reverse_fused(test_windows, 32, 32, 16, 3)
replacement_reverse = reverse_for_export(test_windows, 32, 32, 16, 3)
tf.debugging.assert_equal(original_reverse, replacement_reverse)

print("Partition: exact match.")
print("Reverse: exact match.")

Partition: exact match.
Reverse: exact match.


In [2]:
# Compare the original and modified SwinV2 branch on the same input.
# Both versions use the same pretrained weights and inference settings.

import gc
import numpy as np
import tfswin.window as window_module
import tfswin.winatt as attention_module
from tfswin import SwinTransformerV2Tiny256

test_image = tf.random.stateless_uniform(
    (1, 256, 256, 3),
    seed=(42, 0),
    minval=0,
    maxval=255,
    dtype=tf.float32,
)

original_swin = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)

reference_features = original_swin(
    test_image, training=False
).numpy()

del original_swin
gc.collect()

# Replace both the definitions and the references used by attention.
window_module.window_partition_fused = partition_for_export
window_module.window_reverse_fused = reverse_for_export
attention_module.window_partition_fused = partition_for_export
attention_module.window_reverse_fused = reverse_for_export

swin_branch = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)
swin_branch.trainable = False

updated_features = swin_branch(test_image, training=False).numpy()

print("Feature shape:", updated_features.shape)
print("Maximum difference:",
      float(np.max(np.abs(reference_features - updated_features))))
print("Outputs match:",
      np.allclose(reference_features, updated_features,
                  atol=1e-5, rtol=1e-4))

Feature shape: (1, 8, 8, 768)
Maximum difference: 0.0
Outputs match: True


In [3]:
# Rebuild the hybrid with the modified SwinV2 branch and export it.
# Save a test input and reference output for a separate runtime check.

import keras
from pathlib import Path

keras.utils.set_random_seed(42)

dense_branch = keras.applications.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
)
dense_branch.trainable = False

inputs = keras.Input(shape=(256, 256, 3), name="leaf_image")

dense_maps = dense_branch(
    keras.applications.densenet.preprocess_input(inputs),
    training=False,
)
swin_maps = swin_branch(inputs, training=False)

dense_vector = keras.layers.GlobalAveragePooling2D()(dense_maps)
swin_vector = keras.layers.GlobalAveragePooling2D()(swin_maps)
combined = keras.layers.Concatenate()([dense_vector, swin_vector])
outputs = keras.layers.Dense(
    5, activation="softmax", name="disease_classifier"
)(combined)

hybrid_model = keras.Model(inputs, outputs, name="CassaDx_baseline")

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

np.save(export_dir / "export_test_input.npy", test_image.numpy())
np.save(
    export_dir / "export_test_reference.npy",
    hybrid_model(test_image, training=False).numpy(),
)

saved_path = export_dir / "hybrid_lowrank_export_check"
hybrid_model.export(str(saved_path))

converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_path))
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

converted = converter.convert()
output_path = export_dir / "hybrid_lowrank_flex_fp32.tflite"
output_path.write_bytes(converted)

print("\nModified hybrid exported.")
print("File:", output_path)
print("Size (MiB):", round(len(converted) / (1024 ** 2), 2))

INFO:tensorflow:Assets written to: models/hybrid_lowrank_export_check/assets


INFO:tensorflow:Assets written to: models/hybrid_lowrank_export_check/assets


Saved artifact at 'models/hybrid_lowrank_export_check'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='leaf_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  5139452688: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5133338960: TensorSpec(shape=(3,), dtype=tf.float32, name=None)
  5128076752: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5128076944: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5128075984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5128077136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5128076176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5128073680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5131749712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5131750096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5131748944: TensorSp

W0000 00:00:1792156728.135809 11814835 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792156728.136518 11814835 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 18:18:48.138026: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_lowrank_export_check
2026-10-16 18:18:48.157872: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 18:18:48.157890: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_lowrank_export_check
2026-10-16 18:18:48.393093: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-10-16 18:18:48.450282: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 18:18:49.806311: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_lowrank_export_check
2026-10-1


Modified hybrid exported.
File: models/hybrid_lowrank_flex_fp32.tflite
Size (MiB): 143.27


In [4]:
# Test a replacement for Roll that selects values in their shifted order.
# This checks the replacement without changing the model.

import tensorflow as tf
import numpy as np

def roll_for_export(inputs, shifts, axes):
    outputs = inputs

    for shift, axis in zip(shifts, axes):
        length = tf.shape(outputs)[axis]
        shift = tf.cast(shift, tf.int32)

        indices = tf.math.floormod(
            tf.range(length) - shift,
            length,
        )
        outputs = tf.gather(outputs, indices, axis=axis)

    return outputs


# Check forward shifts, reverse shifts, and no shift.
sample = tf.reshape(
    tf.range(2 * 8 * 8 * 3, dtype=tf.float32),
    (2, 8, 8, 3),
)

for shift in [-4, 0, 4]:
    expected = tf.roll(sample, [shift, shift], [1, 2])
    actual = roll_for_export(sample, [shift, shift], [1, 2])

    np.testing.assert_array_equal(expected.numpy(), actual.numpy())
    print(f"Shift {shift}: exact match.")

Shift -4: exact match.
Shift 0: exact match.
Shift 4: exact match.


In [5]:
# Apply the Roll replacement only inside Swin's module.
# Compare the complete branch before and after the change.

import tfswin.swin as swin_module
from tfswin import SwinTransformerV2Tiny256

# Save the current branch's output before applying the replacement.
reference_features = swin_branch(
    test_image, training=False
).numpy()

# Preserve all other operations and replace only Roll.
class SwinExportOps:
    def __init__(self, original_ops):
        self.original_ops = original_ops

    def __getattr__(self, name):
        return getattr(self.original_ops, name)

    def roll(self, inputs, shift, axis):
        return roll_for_export(inputs, shift, axis)


swin_module.ops = SwinExportOps(swin_module.ops)

# Build a new branch so the replacement is included in its graph.
export_swin = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)
export_swin.trainable = False

updated_features = export_swin(
    test_image, training=False
).numpy()

print("Feature shape:", updated_features.shape)
print(
    "Maximum difference:",
    float(np.max(np.abs(reference_features - updated_features))),
)

np.testing.assert_allclose(
    reference_features,
    updated_features,
    atol=1e-5,
    rtol=1e-5,
)

print("Check passed: Swin features match.")

# Use the checked branch for the next export.
swin_branch = export_swin

Feature shape: (1, 8, 8, 768)
Maximum difference: 0.0
Check passed: Swin features match.


In [6]:
# Rebuild the hybrid with the checked Swin branch, keeping its classifier.
# Verify matching predictions, then attempt export without Flex.

import keras
from pathlib import Path

reference = hybrid_model(test_image, training=False).numpy()

inputs = keras.Input(shape=(256, 256, 3), name="leaf_image")

dense_maps = dense_branch(
    keras.applications.densenet.preprocess_input(inputs),
    training=False,
)
swin_maps = swin_branch(inputs, training=False)

dense_vector = keras.layers.GlobalAveragePooling2D()(dense_maps)
swin_vector = keras.layers.GlobalAveragePooling2D()(swin_maps)

combined = keras.layers.Concatenate()([dense_vector, swin_vector])

# Reuse the existing classifier rather than creating new random weights.
classifier = hybrid_model.get_layer("disease_classifier")
outputs = classifier(combined)

export_model = keras.Model(
    inputs, outputs, name="CassaDx_export"
)

updated = export_model(test_image, training=False).numpy()

np.testing.assert_allclose(
    reference, updated, atol=1e-5, rtol=1e-5
)
print("Hybrid predictions match.")

# Save matching input/output files for the separate runtime test.
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

np.save(model_dir / "builtin_test_input.npy", test_image.numpy())
np.save(model_dir / "builtin_test_reference.npy", updated)

saved_path = model_dir / "hybrid_builtin_export_check"
export_model.export(str(saved_path))

# Allow standard TFLite operations only.
converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_path))
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

converted = converter.convert()

output_path = model_dir / "hybrid_builtin_fp32.tflite"
output_path.write_bytes(converted)

print("Export without Flex succeeded.")
print("Saved:", output_path)
print("Size (MiB):", round(len(converted) / (1024 ** 2), 2))

Hybrid predictions match.
INFO:tensorflow:Assets written to: models/hybrid_builtin_export_check/assets


INFO:tensorflow:Assets written to: models/hybrid_builtin_export_check/assets


Saved artifact at 'models/hybrid_builtin_export_check'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='leaf_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  5185534672: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5185533328: TensorSpec(shape=(3,), dtype=tf.float32, name=None)
  5168732432: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5168728208: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5168727248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168728016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168731856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168728784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168729936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168731280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168733008: TensorSp

W0000 00:00:1792158357.246150 11814835 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792158357.246519 11814835 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 18:45:57.248180: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_builtin_export_check
2026-10-16 18:45:57.267687: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 18:45:57.267713: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_builtin_export_check
2026-10-16 18:45:57.535020: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 18:45:58.888683: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_builtin_export_check
2026-10-16 18:45:59.308092: I tensorflow/cc/saved_model/loader.cc:462] SavedModel load for tags { serve }; Status: success: OK. Took 2059913 m

Export without Flex succeeded.
Saved: models/hybrid_builtin_fp32.tflite
Size (MiB): 143.27


In [1]:
# Rebuild the frozen hybrid with both compatibility fixes applied.
from model_setup import build_hybrid_model
from math import prod

hybrid_model = build_hybrid_model()

print("Output shape:", hybrid_model.output_shape)
print("Trainable parameters:", sum(
    prod(variable.shape)
    for variable in hybrid_model.trainable_variables
))

Output shape: (None, 5)
Trainable parameters: 8965


In [2]:
# Create the hybrid model using our saved setup and compatibility fixes.
from model_setup import build_hybrid_model

hybrid_model = build_hybrid_model()

print("Model input shape:", hybrid_model.input_shape)
print("Model output shape:", hybrid_model.output_shape)

Model input shape: (None, 256, 256, 3)
Model output shape: (None, 5)


In [3]:
# Load the saved training and validation lists.
# Images will be loaded in small batches later to limit memory use.

import csv
from pathlib import Path

project_dir = Path.cwd()
data_dir = project_dir / "data"
split_dir = project_dir / "splits"

def read_split(filename):
    with (split_dir / filename).open(newline="") as file:
        return list(csv.DictReader(file))

train_records = read_split("train.csv")
validation_records = read_split("validation.csv")

assert train_records, "The training split is empty."
assert validation_records, "The validation split is empty."
assert (data_dir / "train_images").is_dir(), "Image folder not found."

print("Training images:", len(train_records))
print("Validation images:", len(validation_records))
print("Model input shape:", hybrid_model.input_shape)
print("Model output shape:", hybrid_model.output_shape)

Training images: 14978
Validation images: 3210
Model input shape: (None, 256, 256, 3)
Model output shape: (None, 5)


In [4]:
# Load images in small batches using our existing preprocessing choices.
# Shuffle and randomly flip training images; keep validation unchanged.

import random
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

BATCH_SIZE = 2

def make_dataset(records, training=False):
    rng = random.Random(42)

    def generate():
        ordered_records = list(records)

        if training:
            rng.shuffle(ordered_records)

        for record in ordered_records:
            image_path = data_dir / "train_images" / record["image_id"]

            with Image.open(image_path) as image:
                image = image.convert("RGB")
                image = image.resize(
                    (256, 256),
                    resample=Image.Resampling.BILINEAR,
                )

                if training and rng.random() < 0.5:
                    image = ImageOps.mirror(image)

                pixels = np.array(image, dtype=np.float32)

            yield pixels, np.int32(record["label"])

    dataset = tf.data.Dataset.from_generator(
        generate,
        output_signature=(
            tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    )

    return dataset.batch(BATCH_SIZE).prefetch(1)


train_dataset = make_dataset(train_records, training=True)
validation_dataset = make_dataset(validation_records, training=False)

images, labels = next(iter(train_dataset))

print("Image batch shape:", images.shape)
print("Labels:", labels.numpy())
print("Pixel range:", float(tf.reduce_min(images)),
      "to", float(tf.reduce_max(images)))
print("Data type:", images.dtype)

Image batch shape: (2, 256, 256, 3)
Labels: [0 4]
Pixel range: 0.0 to 255.0
Data type: <dtype: 'float32'>


In [5]:
# Check that the model produces five probabilities for each image.
# This runs prediction only; it does not update any weights.

predictions = hybrid_model(images, training=False).numpy()

assert predictions.shape == (2, 5)
assert np.isfinite(predictions).all()
np.testing.assert_allclose(
    predictions.sum(axis=1),
    np.ones(2),
    atol=1e-5,
)

print("Prediction shape:", predictions.shape)
print("All values finite:", bool(np.isfinite(predictions).all()))
print("Probability sums:", predictions.sum(axis=1))
print("Batch check passed.")

Prediction shape: (2, 5)
All values finite: True
Probability sums: [1. 1.]
Batch check passed.


In [6]:
# Calculate class weights using only the training split.
# Give less common classes more weight in the training loss.

from collections import Counter

class_counts = Counter(int(row["label"]) for row in train_records)
number_of_classes = 5
total_images = len(train_records)

assert set(class_counts) == set(range(number_of_classes))

class_weights = {
    label: total_images / (number_of_classes * count)
    for label, count in sorted(class_counts.items())
}

for label, weight in class_weights.items():
    print(f"Class {label}: {weight:.3f}")
    

Class 0: 3.936
Class 1: 1.955
Class 2: 1.794
Class 3: 0.325
Class 4: 1.661


In [7]:
# Configure training for the five-class classifier.
# Integer labels (0–4) use sparse categorical cross-entropy.

import keras
from math import prod

trainable_count = sum(
    prod(variable.shape)
    for variable in hybrid_model.trainable_variables
)

assert trainable_count == 8965, (
    f"Expected 8,965 trainable parameters, found {trainable_count}."
)

hybrid_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

print("Trainable parameters:", trainable_count)
print("Model configured for training.")

Trainable parameters: 8965
Model configured for training.


In [8]:
# Train on one batch and measure how long it takes.
# Apply class weights while keeping both pretrained branches frozen.

from time import perf_counter

start = perf_counter()

batch_result = hybrid_model.train_on_batch(
    images,
    labels,
    class_weight=class_weights,
    return_dict=True,
)

elapsed = perf_counter() - start

print("Batch size:", int(images.shape[0]))
print("Time (seconds):", round(elapsed, 2))

for name, value in batch_result.items():
    print(f"{name}: {float(value):.4f}")

# Clear metric averages before the later training run.
# This does not undo the weight update.
hybrid_model.reset_metrics()

Batch size: 2
Time (seconds): 11.77
accuracy: 0.0000
loss: 3.5946


In [9]:
# Time three additional training batches after the initial setup.
# These steps update only the classifier and apply our class weights.

batch_times = []

for step, (batch_images, batch_labels) in enumerate(
    train_dataset.take(3), start=1
):
    start = perf_counter()

    result = hybrid_model.train_on_batch(
        batch_images,
        batch_labels,
        class_weight=class_weights,
        return_dict=True,
    )

    elapsed = perf_counter() - start
    batch_times.append(elapsed)
    print(f"Batch {step}: {elapsed:.2f} seconds")

average_seconds = sum(batch_times) / len(batch_times)
batches_per_epoch = (
    len(train_records) + BATCH_SIZE - 1
) // BATCH_SIZE

print("Average seconds per batch:", round(average_seconds, 2))
print(
    "Estimated training-only hours per epoch:",
    round(average_seconds * batches_per_epoch / 3600, 2),
)


hybrid_model.reset_metrics()


Batch 1: 2.61 seconds
Batch 2: 2.37 seconds
Batch 3: 2.54 seconds
Average seconds per batch: 2.51
Estimated training-only hours per epoch: 5.22


2026-10-19 15:02:49.407090: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [10]:
# Inspect the combined features that feed the classifier.
# This does not train the model or change its weights.

classifier = hybrid_model.get_layer("disease_classifier")

feature_extractor = keras.Model(
    inputs=hybrid_model.input,
    outputs=classifier.input,
    name="cassadx_feature_extractor",
)

print("Feature shape:", feature_extractor.output_shape)
print("Trainable feature variables:",
      len(feature_extractor.trainable_variables))

Feature shape: (None, 1792)
Trainable feature variables: 0


In [11]:
# Prepare deterministic training images for feature extraction.
# Keep their saved order and do not apply random augmentation.

from pathlib import Path

feature_dir = Path("features")
feature_dir.mkdir(exist_ok=True)

train_feature_dataset = make_dataset(
    train_records,
    training=False,
)

print("Feature folder:", feature_dir.resolve())
print("Training images to extract:", len(train_records))
print("Features per image:", feature_extractor.output_shape[-1])
print("Random augmentation: disabled for this extraction")

Feature folder: /Users/hf/Documents/CassaDx/features
Training images to extract: 14978
Features per image: 1792
Random augmentation: disabled for this extraction


In [12]:
# Extract and save features for the first 100 training images.
# Save filenames and labels alongside features to preserve their alignment.

from time import perf_counter
import numpy as np

chunk_records = train_records[:100]
chunk_dataset = make_dataset(chunk_records, training=False)
chunk_path = feature_dir / "train_00000_00100.npz"

feature_batches = []
start = perf_counter()
processed = 0

for batch_images, batch_labels in chunk_dataset:
    batch_features = feature_extractor(
        batch_images, training=False
    ).numpy()

    assert np.isfinite(batch_features).all()
    feature_batches.append(batch_features)

    processed += len(batch_features)
    if processed % 20 == 0:
        print(f"Processed {processed}/100 images...", flush=True)

features = np.concatenate(feature_batches, axis=0)

filenames = np.array([
    row["image_id"] for row in chunk_records
])
saved_labels = np.array([
    int(row["label"]) for row in chunk_records
], dtype=np.int32)

assert features.shape == (100, 1792)

np.savez(
    chunk_path,
    features=features,
    labels=saved_labels,
    filenames=filenames,
)

# Reopen the saved file and verify its contents.
with np.load(chunk_path, allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved["features"], features)
    np.testing.assert_array_equal(saved["labels"], saved_labels)
    np.testing.assert_array_equal(saved["filenames"], filenames)

print("Saved and verified:", chunk_path)
print("Feature shape:", features.shape)
print("Elapsed minutes:", round((perf_counter() - start) / 60, 2))

Processed 20/100 images...
Processed 40/100 images...
Processed 60/100 images...
Processed 80/100 images...
Processed 100/100 images...
Saved and verified: features/train_00000_00100.npz
Feature shape: (100, 1792)
Elapsed minutes: 0.71


2026-10-19 15:05:26.914964: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [13]:
# Cache training and validation features in chunks of 100 images.
# Verify existing chunks and skip them when resuming.

def verify_feature_chunk(path, records):
    expected_names = np.array([r["image_id"] for r in records])
    expected_labels = np.array(
        [int(r["label"]) for r in records], dtype=np.int32
    )

    with np.load(path, allow_pickle=False) as saved:
        assert saved["features"].shape == (len(records), 1792)
        assert np.isfinite(saved["features"]).all()
        np.testing.assert_array_equal(saved["filenames"], expected_names)
        np.testing.assert_array_equal(saved["labels"], expected_labels)


def cache_split_features(records, split_name):
    started = perf_counter()

    for start in range(0, len(records), 100):
        end = min(start + 100, len(records))
        chunk_records = records[start:end]
        path = feature_dir / f"{split_name}_{start:05d}_{end:05d}.npz"

        if path.exists():
            verify_feature_chunk(path, chunk_records)
            print(f"{split_name}: verified existing {end}/{len(records)}",
                  flush=True)
            continue

        batches = []
        dataset = make_dataset(chunk_records, training=False)

        for batch_images, _ in dataset:
            values = feature_extractor(
                batch_images, training=False
            ).numpy()
            assert np.isfinite(values).all()
            batches.append(values)

        features = np.concatenate(batches, axis=0)
        assert features.shape == (len(chunk_records), 1792)

        # Write to a temporary file before marking this chunk complete.
        temporary_path = path.with_suffix(".tmp")
        with temporary_path.open("wb") as file:
            np.savez(
                file,
                features=features,
                labels=np.array(
                    [int(r["label"]) for r in chunk_records],
                    dtype=np.int32,
                ),
                filenames=np.array(
                    [r["image_id"] for r in chunk_records]
                ),
            )

        verify_feature_chunk(temporary_path, chunk_records)
        temporary_path.replace(path)

        minutes = (perf_counter() - started) / 60
        print(
            f"{split_name}: saved {end}/{len(records)} "
            f"— {minutes:.1f} minutes elapsed",
            flush=True,
        )

    print(f"{split_name}: complete.", flush=True)


cache_split_features(train_records, "train")
cache_split_features(validation_records, "validation")

train: verified existing 100/14978
train: saved 200/14978 — 0.7 minutes elapsed
train: saved 300/14978 — 1.4 minutes elapsed


2026-10-19 15:08:00.175145: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 400/14978 — 2.0 minutes elapsed
train: saved 500/14978 — 2.7 minutes elapsed
train: saved 600/14978 — 3.4 minutes elapsed
train: saved 700/14978 — 4.0 minutes elapsed


2026-10-19 15:10:40.717080: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 800/14978 — 4.7 minutes elapsed
train: saved 900/14978 — 5.4 minutes elapsed
train: saved 1000/14978 — 6.1 minutes elapsed
train: saved 1100/14978 — 6.8 minutes elapsed
train: saved 1200/14978 — 7.5 minutes elapsed
train: saved 1300/14978 — 8.2 minutes elapsed
train: saved 1400/14978 — 8.8 minutes elapsed
train: saved 1500/14978 — 9.5 minutes elapsed


2026-10-19 15:24:52.674046: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 1600/14978 — 10.2 minutes elapsed
train: saved 1700/14978 — 10.8 minutes elapsed
train: saved 1800/14978 — 11.5 minutes elapsed
train: saved 1900/14978 — 12.2 minutes elapsed
train: saved 2000/14978 — 12.9 minutes elapsed
train: saved 2100/14978 — 13.5 minutes elapsed
train: saved 2200/14978 — 14.2 minutes elapsed
train: saved 2300/14978 — 14.9 minutes elapsed
train: saved 2400/14978 — 15.5 minutes elapsed
train: saved 2500/14978 — 16.2 minutes elapsed
train: saved 2600/14978 — 16.9 minutes elapsed
train: saved 2700/14978 — 17.5 minutes elapsed
train: saved 2800/14978 — 18.2 minutes elapsed
train: saved 2900/14978 — 18.9 minutes elapsed
train: saved 3000/14978 — 19.6 minutes elapsed
train: saved 3100/14978 — 20.2 minutes elapsed


2026-10-19 15:37:50.419262: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 3200/14978 — 20.9 minutes elapsed
train: saved 3300/14978 — 21.6 minutes elapsed
train: saved 3400/14978 — 22.3 minutes elapsed
train: saved 3500/14978 — 22.9 minutes elapsed


KeyboardInterrupt: 

In [14]:
# Resume extraction and skip completed chunks.
cache_split_features(train_records, "train")
cache_split_features(validation_records, "validation")

train: verified existing 100/14978
train: verified existing 200/14978
train: verified existing 300/14978
train: verified existing 400/14978
train: verified existing 500/14978
train: verified existing 600/14978
train: verified existing 700/14978
train: verified existing 800/14978
train: verified existing 900/14978
train: verified existing 1000/14978
train: verified existing 1100/14978
train: verified existing 1200/14978
train: verified existing 1300/14978
train: verified existing 1400/14978
train: verified existing 1500/14978
train: verified existing 1600/14978
train: verified existing 1700/14978
train: verified existing 1800/14978
train: verified existing 1900/14978
train: verified existing 2000/14978
train: verified existing 2100/14978
train: verified existing 2200/14978
train: verified existing 2300/14978
train: verified existing 2400/14978
train: verified existing 2500/14978
train: verified existing 2600/14978
train: verified existing 2700/14978
train: verified existing 2800/14978
t

KeyboardInterrupt: 

In [1]:
# Check GPU detection in a newly started notebook session.
import os
import sys
import tensorflow as tf

print("Process ID:", os.getpid())
print("Python:", sys.executable)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

Process ID: 5776
Python: /Users/hf/Documents/CassaDx/.venv-train/bin/python
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Rebuild the frozen hybrid and expose its combined image features.
# This does not train the model or resume extraction.

import keras
from model_setup import build_hybrid_model

hybrid_model = build_hybrid_model()

feature_extractor = keras.Model(
    inputs=hybrid_model.input,
    outputs=hybrid_model.get_layer("disease_classifier").input,
    name="cassadx_feature_extractor",
)

print("Feature shape:", feature_extractor.output_shape)
print("Trainable feature variables:",
      len(feature_extractor.trainable_variables))

2026-10-19 17:42:13.717847: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-10-19 17:42:13.717880: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-10-19 17:42:13.717887: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-10-19 17:42:13.717902: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-19 17:42:13.717913: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Feature shape: (None, 1792)
Trainable feature variables: 0


In [3]:
# Compare GPU features with saved CPU features for the same two images.
# Repeat inference once to measure speed after warm-up.

from pathlib import Path
from time import perf_counter
from PIL import Image
import numpy as np

project_dir = Path.cwd()
feature_dir = project_dir / "features"

with np.load(
    feature_dir / "train_00000_00100.npz",
    allow_pickle=False,
) as saved:
    filenames = saved["filenames"][:2].copy()
    cpu_features = saved["features"][:2].copy()

images = []
for filename in filenames:
    with Image.open(
        project_dir / "data" / "train_images" / str(filename)
    ) as image:
        image = image.convert("RGB").resize(
            (256, 256),
            resample=Image.Resampling.BILINEAR,
        )
        images.append(np.array(image, dtype=np.float32))

image_batch = np.stack(images)

# First pass warms up execution.
gpu_features = feature_extractor(
    image_batch, training=False
).numpy()

# Converting to NumPy waits for computation to finish.
start = perf_counter()
gpu_features = feature_extractor(
    image_batch, training=False
).numpy()
elapsed = perf_counter() - start

difference = np.abs(cpu_features - gpu_features)

print("All values finite:", bool(np.isfinite(gpu_features).all()))
print("Maximum difference:", float(difference.max()))
print("Mean difference:", float(difference.mean()))
print("Features close:", np.allclose(
    cpu_features, gpu_features, atol=1e-4, rtol=1e-3
))
print("Seconds for 2 images:", round(elapsed, 3))

All values finite: True
Maximum difference: 3.5762786865234375e-06
Mean difference: 3.537658699315216e-07
Features close: True
Seconds for 2 images: 1.124


In [4]:
# Compile feature extraction and time three runs after warm-up.
# Confirm that compilation preserves the saved CPU features.

import tensorflow as tf

@tf.function(
    input_signature=[
        tf.TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32)
    ]
)
def extract_features_fast(batch):
    return feature_extractor(batch, training=False)

# Warm up tracing and GPU execution before measuring.
compiled_features = extract_features_fast(image_batch).numpy()

times = []
for _ in range(3):
    start = perf_counter()
    compiled_features = extract_features_fast(image_batch).numpy()
    times.append(perf_counter() - start)

print("All values finite:",
      bool(np.isfinite(compiled_features).all()))
print("Features close:", np.allclose(
    cpu_features, compiled_features, atol=1e-4, rtol=1e-3
))
print("Times for 2 images:", [round(t, 3) for t in times])
print("Average seconds:", round(sum(times) / len(times), 3))

2026-10-19 17:43:50.470385: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


All values finite: True
Features close: True
Times for 2 images: [0.469, 0.254, 0.181]
Average seconds: 0.301


In [5]:
# Resume feature extraction using the tested compiled GPU function.
# Keep preprocessing unchanged and verify saved chunks before skipping them.

import csv
from pathlib import Path
from time import perf_counter

import numpy as np
import tensorflow as tf
from PIL import Image

assert callable(globals().get("extract_features_fast")), (
    "Run the compiled feature-extraction cell first."
)

project_dir = Path.cwd()
data_dir = project_dir / "data"
split_dir = project_dir / "splits"
feature_dir = project_dir / "features"
feature_dir.mkdir(exist_ok=True)

BATCH_SIZE = 2
CHUNK_SIZE = 100

def read_split(filename):
    with (split_dir / filename).open(newline="") as file:
        return list(csv.DictReader(file))

train_records = read_split("train.csv")
validation_records = read_split("validation.csv")


def make_feature_dataset(records):
    # Preserve filename order and apply no random augmentation.
    def generate():
        for record in records:
            path = data_dir / "train_images" / record["image_id"]

            with Image.open(path) as image:
                image = image.convert("RGB").resize(
                    (256, 256),
                    resample=Image.Resampling.BILINEAR,
                )
                pixels = np.array(image, dtype=np.float32)

            yield pixels

    dataset = tf.data.Dataset.from_generator(
        generate,
        output_signature=tf.TensorSpec(
            shape=(256, 256, 3),
            dtype=tf.float32,
        ),
    )

    return dataset.batch(BATCH_SIZE).prefetch(1)


def verify_feature_chunk(path, records):
    expected_names = np.array([r["image_id"] for r in records])
    expected_labels = np.array(
        [int(r["label"]) for r in records],
        dtype=np.int32,
    )

    with np.load(path, allow_pickle=False) as saved:
        assert saved["features"].shape == (len(records), 1792)
        assert np.isfinite(saved["features"]).all()
        np.testing.assert_array_equal(
            saved["filenames"], expected_names
        )
        np.testing.assert_array_equal(
            saved["labels"], expected_labels
        )


def cache_split_features(records, split_name):
    started = perf_counter()
    skipped = 0
    extracted = 0

    for start in range(0, len(records), CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, len(records))
        chunk_records = records[start:end]
        path = feature_dir / f"{split_name}_{start:05d}_{end:05d}.npz"

        if path.exists():
            verify_feature_chunk(path, chunk_records)
            skipped += len(chunk_records)
            continue

        if skipped:
            print(
                f"{split_name}: verified and skipped "
                f"{skipped} saved images.",
                flush=True,
            )
            skipped = 0

        batches = []

        for batch_images in make_feature_dataset(chunk_records):
            values = extract_features_fast(batch_images).numpy()
            assert np.isfinite(values).all()
            batches.append(values)

        features = np.concatenate(batches, axis=0)
        assert features.shape == (len(chunk_records), 1792)

        # Only mark a chunk complete after saving and verifying it.
        temporary_path = path.with_suffix(".tmp")

        with temporary_path.open("wb") as file:
            np.savez(
                file,
                features=features,
                labels=np.array(
                    [int(r["label"]) for r in chunk_records],
                    dtype=np.int32,
                ),
                filenames=np.array(
                    [r["image_id"] for r in chunk_records]
                ),
            )

        verify_feature_chunk(temporary_path, chunk_records)
        temporary_path.replace(path)

        extracted += len(chunk_records)
        minutes = (perf_counter() - started) / 60

        print(
            f"{split_name}: saved {end}/{len(records)} "
            f"— {minutes:.1f} minutes elapsed",
            flush=True,
        )

    if skipped:
        print(
            f"{split_name}: verified and skipped "
            f"{skipped} saved images.",
            flush=True,
        )

    print(
        f"{split_name}: complete. "
        f"{extracted} new images extracted this run.",
        flush=True,
    )


print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("Training images:", len(train_records))
print("Validation images:", len(validation_records))

cache_split_features(train_records, "train")
cache_split_features(validation_records, "validation")

print("Training and validation feature extraction finished.")

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training images: 14978
Validation images: 3210
train: verified and skipped 4500 saved images.
train: saved 4600/14978 — 0.3 minutes elapsed


2026-10-19 17:45:33.489087: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 4700/14978 — 0.4 minutes elapsed


2026-10-19 17:45:42.881209: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 4800/14978 — 0.6 minutes elapsed
train: saved 4900/14978 — 0.8 minutes elapsed


...

train: saved 7800/14978 — 5.1 minutes elapsed
train: saved 7900/14978 — 5.2 minutes elapsed
train: saved 8000/14978 — 5.4 minutes elapsed
train: saved 8100/14978 — 5.5 minutes elapsed
train: saved 8200/14978 — 5.7 minutes elapsed
train: saved 8300/14978 — 5.8 minutes elapsed
train: saved 8400/14978 — 6.0 minutes elapsed
train: saved 8500/14978 — 6.1 minutes elapsed
train: saved 8600/14978 — 6.3 minutes elapsed
train: saved 8700/14978 — 6.4 minutes elapsed
train: saved 8800/14978 — 6.6 minutes elapsed
train: saved 8900/14978 — 6.7 minutes elapsed
train: saved 9000/14978 — 6.9 minutes elapsed
train: saved 9100/14978 — 7.0 minutes elapsed
train: saved 9200/14978 — 7.2 minutes elapsed
train: saved 9300/14978 — 7.4 minutes elapsed
train: saved 9400/14978 — 7.5 minutes elapsed
train: saved 9500/14978 — 7.7 minutes elapsed
train: saved 9600/14978 — 7.8 minutes elapsed
train: saved 9700/14978 — 8.0 minutes elapsed
train: saved 9800/14978 — 8.1 minutes elapsed
train: saved 9900/14978 — 8.3 minu

2026-10-19 17:55:02.304923: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train: saved 11000/14978 — 9.9 minutes elapsed
train: saved 11100/14978 — 10.1 minutes elapsed
train: saved 11200/14978 — 10.2 minutes elapsed
train: saved 11300/14978 — 10.4 minutes elapsed
train: saved 11400/14978 — 10.5 minutes elapsed
train: saved 11500/14978 — 10.6 minutes elapsed
train: saved 11600/14978 — 10.8 minutes elapsed
train: saved 11700/14978 — 10.9 minutes elapsed
train: saved 11800/14978 — 11.1 minutes elapsed
train: saved 11900/14978 — 11.2 minutes elapsed
train: saved 12000/14978 — 11.4 minutes elapsed
train: saved 12100/14978 — 11.5 minutes elapsed
train: saved 12200/14978 — 11.7 minutes elapsed
train: saved 12300/14978 — 11.8 minutes elapsed
train: saved 12400/14978 — 12.0 minutes elapsed
train: saved 12500/14978 — 12.1 minutes elapsed
train: saved 12600/14978 — 12.2 minutes elapsed
train: saved 12700/14978 — 12.4 minutes elapsed
train: saved 12800/14978 — 12.5 minutes elapsed
train: saved 12900/14978 — 12.7 minutes elapsed
train: saved 13000/14978 — 12.8 minutes e

2026-10-19 18:04:21.832239: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


validation: saved 2400/3210 — 3.6 minutes elapsed
validation: saved 2500/3210 — 3.7 minutes elapsed
validation: saved 2600/3210 — 3.9 minutes elapsed
validation: saved 2700/3210 — 4.0 minutes elapsed
validation: saved 2800/3210 — 4.1 minutes elapsed
validation: saved 2900/3210 — 4.3 minutes elapsed
validation: saved 3000/3210 — 4.4 minutes elapsed
validation: saved 3100/3210 — 4.6 minutes elapsed
validation: saved 3200/3210 — 4.7 minutes elapsed
validation: saved 3210/3210 — 4.7 minutes elapsed
validation: complete. 3210 new images extracted this run.
Training and validation feature extraction finished.


In [6]:
# Load cached features in split order and verify their labels and filenames.
# These arrays will be used to train and validate the small classifier.

def load_cached_features(records, split_name):
    feature_parts = []
    label_parts = []

    for start in range(0, len(records), CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, len(records))
        chunk_records = records[start:end]
        path = feature_dir / f"{split_name}_{start:05d}_{end:05d}.npz"

        verify_feature_chunk(path, chunk_records)

        with np.load(path, allow_pickle=False) as saved:
            feature_parts.append(saved["features"].copy())
            label_parts.append(saved["labels"].copy())

    return (
        np.concatenate(feature_parts, axis=0),
        np.concatenate(label_parts, axis=0),
    )


train_features, train_labels = load_cached_features(
    train_records, "train"
)
validation_features, validation_labels = load_cached_features(
    validation_records, "validation"
)

assert train_features.shape == (14978, 1792)
assert validation_features.shape == (3210, 1792)

print("Training features:", train_features.shape)
print("Training labels:", train_labels.shape)
print("Validation features:", validation_features.shape)
print("Validation labels:", validation_labels.shape)
print("Checks passed: cached features match the saved splits.")

Training features: (14978, 1792)
Training labels: (14978,)
Validation features: (3210, 1792)
Validation labels: (3210,)
Checks passed: cached features match the saved splits.


In [7]:
# Create a fresh five-class classifier for the cached features.
# Recalculate training class weights and configure learning.

import keras
import numpy as np
from math import prod

keras.utils.set_random_seed(42)

feature_input = keras.Input(shape=(1792,), name="cached_features")
predictions = keras.layers.Dense(
    5,
    activation="softmax",
    name="disease_classifier",
)(feature_input)

classifier_model = keras.Model(
    feature_input,
    predictions,
    name="cassadx_cached_classifier",
)

counts = np.bincount(train_labels, minlength=5)
assert np.all(counts > 0)

class_weights = {
    label: float(len(train_labels) / (5 * count))
    for label, count in enumerate(counts)
}

classifier_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ],
)

print("Input shape:", classifier_model.input_shape)
print("Output shape:", classifier_model.output_shape)
print("Trainable parameters:", sum(
    prod(variable.shape)
    for variable in classifier_model.trainable_variables
))
print("Classifier ready.")

Input shape: (None, 1792)
Output shape: (None, 5)
Trainable parameters: 8965
Classifier ready.


In [8]:
# Train the classifier with class weights.
# Save the best validation-loss checkpoint and stop if progress stalls.

from pathlib import Path

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

best_weights_path = (
    checkpoint_dir / "cached_classifier_best.weights.h5"
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(best_weights_path),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.TerminateOnNaN(),
]

history = classifier_model.fit(
    train_features,
    train_labels,
    validation_data=(validation_features, validation_labels),
    class_weight=class_weights,
    batch_size=64,
    epochs=30,
    shuffle=True,
    callbacks=callbacks,
    verbose=2,
)

Epoch 1/30

Epoch 1: val_loss improved from inf to 0.77169, saving model to checkpoints/cached_classifier_best.weights.h5
235/235 - 5s - 20ms/step - accuracy: 0.6363 - loss: 1.0765 - val_accuracy: 0.7174 - val_loss: 0.7717 - learning_rate: 0.0010
Epoch 2/30

Epoch 2: val_loss improved from 0.77169 to 0.73629, saving model to checkpoints/cached_classifier_best.weights.h5
235/235 - 2s - 7ms/step - accuracy: 0.7253 - loss: 0.8791 - val_accuracy: 0.7346 - val_loss: 0.7363 - learning_rate: 0.0010
Epoch 3/30

Epoch 3: val_loss improved from 0.73629 to 0.71862, saving model to checkpoints/cached_classifier_best.weights.h5
235/235 - 2s - 7ms/step - accuracy: 0.7506 - loss: 0.8109 - val_accuracy: 0.7408 - val_loss: 0.7186 - learning_rate: 0.0010
Epoch 4/30

Epoch 4: val_loss improved from 0.71862 to 0.70561, saving model to checkpoints/cached_classifier_best.weights.h5
235/235 - 2s - 7ms/step - accuracy: 0.7629 - loss: 0.7710 - val_accuracy: 0.7455 - val_loss: 0.7056 - learning_rate: 0.0010
Epo

In [9]:
# Evaluate each class on validation data using the best saved classifier.
# Calculate precision, recall, and F1 without installing extra packages.

classifier_model.load_weights(str(best_weights_path))

probabilities = classifier_model.predict(
    validation_features, batch_size=64, verbose=0
)
predicted_labels = np.argmax(probabilities, axis=1)

confusion = np.zeros((5, 5), dtype=np.int64)
np.add.at(confusion, (validation_labels, predicted_labels), 1)

names = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
f1_scores = []
recalls = []

print("Class      Precision   Recall      F1     Images")

for index, name in enumerate(names):
    correct = confusion[index, index]
    actual = confusion[index].sum()
    predicted = confusion[:, index].sum()

    precision = correct / predicted if predicted else 0.0
    recall = correct / actual if actual else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    recalls.append(recall)
    f1_scores.append(f1)

    print(
        f"{name:<10} {precision:>8.2%}  {recall:>8.2%}  "
        f"{f1:>8.2%}  {actual:>6}"
    )

print("\nValidation accuracy:",
      f"{np.mean(predicted_labels == validation_labels):.2%}")
print("Macro F1:", f"{np.mean(f1_scores):.2%}")
print("Balanced accuracy:", f"{np.mean(recalls):.2%}")
print("\nConfusion matrix: rows = actual, columns = predicted")
print("Class order:", names)
print(confusion)

Class      Precision   Recall      F1     Images
CBB          44.74%    52.15%    48.16%     163
CBSD         58.90%    71.65%    64.65%     328
CGM          67.01%    72.63%    69.71%     358
CMD          94.89%    87.49%    91.04%    1974
Healthy      59.32%    63.31%    61.25%     387

Validation accuracy: 79.50%
Macro F1: 66.96%
Balanced accuracy: 69.44%

Confusion matrix: rows = actual, columns = predicted
Class order: ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']
[[  85   31    3   11   33]
 [  23  235   15   13   42]
 [  12   21  260   38   27]
 [  20   72   89 1727   66]
 [  50   40   21   31  245]]


In [10]:
# Save validation metrics and predictions for future comparisons.
# This does not change the model or access the test split.

import json
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

metrics = {
    "experiment": "frozen_hybrid_unflipped_features",
    "split": "validation",
    "selected_epoch": 24,
    "accuracy": float(np.mean(predicted_labels == validation_labels)),
    "macro_f1": float(np.mean(f1_scores)),
    "balanced_accuracy": float(np.mean(recalls)),
    "class_order": names,
    "per_class_recall": {
        name: float(recalls[i]) for i, name in enumerate(names)
    },
    "per_class_f1": {
        name: float(f1_scores[i]) for i, name in enumerate(names)
    },
    "confusion_matrix": confusion.tolist(),
}

with (results_dir / "frozen_baseline_validation.json").open("w") as file:
    json.dump(metrics, file, indent=2)

np.savez(
    results_dir / "frozen_baseline_validation_predictions.npz",
    filenames=np.array([r["image_id"] for r in validation_records]),
    actual_labels=validation_labels,
    predicted_labels=predicted_labels,
    probabilities=probabilities,
)

print("Baseline validation results saved.")

Baseline validation results saved.


In [11]:
# Restore the best classifier and copy its learned weights into the hybrid.
# Verify that both models make matching predictions from the same features.

classifier_model.load_weights(str(best_weights_path))

hybrid_model.get_layer("disease_classifier").set_weights(
    classifier_model.get_layer("disease_classifier").get_weights()
)

# Load two validation images using the same preprocessing as extraction.
check_images = []

for record in validation_records[:2]:
    path = data_dir / "train_images" / record["image_id"]

    with Image.open(path) as image:
        image = image.convert("RGB").resize(
            (256, 256),
            resample=Image.Resampling.BILINEAR,
        )
        check_images.append(np.array(image, dtype=np.float32))

check_images = np.stack(check_images)

image_predictions = hybrid_model(
    check_images, training=False
).numpy()

cached_predictions = classifier_model(
    validation_features[:2], training=False
).numpy()

print(
    "Maximum prediction difference:",
    float(np.max(np.abs(image_predictions - cached_predictions))),
)

np.testing.assert_allclose(
    image_predictions,
    cached_predictions,
    atol=1e-4,
    rtol=1e-3,
)

# Preserve the full baseline before changing any trainable layers.
baseline_path = checkpoint_dir / "hybrid_frozen_baseline.weights.h5"
hybrid_model.save_weights(str(baseline_path))

print("Check passed: image and cached-feature predictions match.")
print("Full baseline saved:", baseline_path)

Maximum prediction difference: 5.960464477539062e-07
Check passed: image and cached-feature predictions match.
Full baseline saved: checkpoints/hybrid_frozen_baseline.weights.h5


In [12]:
# Inspect the nested model branches and their final layers.
# This does not change which layers are trainable.

import keras

for branch in hybrid_model.layers:
    if isinstance(branch, keras.Model):
        print(f"\nBranch: {branch.name}")
        print("Number of layers:", len(branch.layers))
        print("Final 20 layers:")

        for layer in branch.layers[-20:]:
            print(f"  {layer.name}: {type(layer).__name__}")


Branch: densenet121
Number of layers: 427
Final 20 layers:
  conv5_block14_1_bn: BatchNormalization
  conv5_block14_1_relu: Activation
  conv5_block14_2_conv: Conv2D
  conv5_block14_concat: Concatenate
  conv5_block15_0_bn: BatchNormalization
  conv5_block15_0_relu: Activation
  conv5_block15_1_conv: Conv2D
  conv5_block15_1_bn: BatchNormalization
  conv5_block15_1_relu: Activation
  conv5_block15_2_conv: Conv2D
  conv5_block15_concat: Concatenate
  conv5_block16_0_bn: BatchNormalization
  conv5_block16_0_relu: Activation
  conv5_block16_1_conv: Conv2D
  conv5_block16_1_bn: BatchNormalization
  conv5_block16_1_relu: Activation
  conv5_block16_2_conv: Conv2D
  conv5_block16_concat: Concatenate
  bn: BatchNormalization
  relu: Activation

Branch: swin2_tiny_256
Number of layers: 12
Final 20 layers:
  input_layer_1: InputLayer
  normalize: Normalization
  patch_embed: PatchEmbedding
  pos_drop: Dropout
  layers.0: BasicLayer
  layers.0.downsample: PatchMerging
  layers.1: BasicLayer
  la

In [13]:
# Unfreeze only DenseNet's last three convolution blocks.
# Keep SwinV2 and BatchNormalization frozen, then recompile.

import keras
from math import prod

dense_branch = hybrid_model.get_layer("densenet121")
swin_branch = hybrid_model.get_layer("swin2_tiny_256")

# Enable the parent branch before selecting its trainable layers.
dense_branch.trainable = True
swin_branch.trainable = False

selected_blocks = (
    "conv5_block14_",
    "conv5_block15_",
    "conv5_block16_",
)

for layer in dense_branch.layers:
    layer.trainable = (
        isinstance(layer, keras.layers.Conv2D)
        and layer.name.startswith(selected_blocks)
    )

hybrid_model.get_layer("disease_classifier").trainable = True

assert not swin_branch.trainable_variables
assert all(
    not layer.trainable
    for layer in dense_branch.layers
    if isinstance(layer, keras.layers.BatchNormalization)
)

# A smaller learning rate limits changes to pretrained weights.
hybrid_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ],
)

print("Trainable DenseNet layers:")
for layer in dense_branch.layers:
    if layer.trainable_weights:
        print(" ", layer.name)

print("Total trainable parameters:", sum(
    prod(variable.shape)
    for variable in hybrid_model.trainable_variables
))
print("SwinV2 remains frozen.")
print("Fine-tuning configured; training has not started.")

Trainable DenseNet layers:
  conv5_block14_1_conv
  conv5_block14_2_conv
  conv5_block15_1_conv
  conv5_block15_2_conv
  conv5_block16_1_conv
  conv5_block16_2_conv
Total trainable parameters: 488197
SwinV2 remains frozen.
Fine-tuning configured; training has not started.


In [14]:
# Load images for fine-tuning, shuffling and flipping training images only.
# Preserve raw 0–255 pixels because normalization is inside the model.

import random
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

FINETUNE_BATCH_SIZE = 2

def make_finetune_dataset(records, training=False):
    rng = random.Random(42)

    def generate():
        ordered_records = list(records)

        if training:
            rng.shuffle(ordered_records)

        for record in ordered_records:
            path = data_dir / "train_images" / record["image_id"]

            with Image.open(path) as image:
                image = image.convert("RGB").resize(
                    (256, 256),
                    resample=Image.Resampling.BILINEAR,
                )

                if training and rng.random() < 0.5:
                    image = ImageOps.mirror(image)

                pixels = np.array(image, dtype=np.float32)

            yield pixels, np.int32(record["label"])

    dataset = tf.data.Dataset.from_generator(
        generate,
        output_signature=(
            tf.TensorSpec((256, 256, 3), tf.float32),
            tf.TensorSpec((), tf.int32),
        ),
    )

    # Declare the image count so training knows when each epoch ends.
    dataset = dataset.apply(
        tf.data.experimental.assert_cardinality(len(records))
    )

    return dataset.batch(FINETUNE_BATCH_SIZE).prefetch(1)


finetune_train_dataset = make_finetune_dataset(
    train_records, training=True
)
finetune_validation_dataset = make_finetune_dataset(
    validation_records, training=False
)

check_images, check_labels = next(iter(finetune_train_dataset))

print("Image batch shape:", check_images.shape)
print("Labels:", check_labels.numpy())
print("Training batches:", len(finetune_train_dataset))
print("Validation batches:", len(finetune_validation_dataset))

Image batch shape: (2, 256, 256, 3)
Labels: [0 4]
Training batches: 7489
Validation batches: 1605


In [15]:
# Test fine-tuning speed, then undo the trial updates.
# Keep the saved baseline as the starting point for real training.

from time import perf_counter

original_weights = [
    variable.numpy().copy()
    for variable in hybrid_model.trainable_variables
]

try:
    print("Running warm-up batch...", flush=True)

    hybrid_model.train_on_batch(
        check_images,
        check_labels,
        class_weight=class_weights,
        return_dict=True,
    )

    timings = []

    for step, (images, labels) in enumerate(
        finetune_train_dataset.take(3), start=1
    ):
        start = perf_counter()

        result = hybrid_model.train_on_batch(
            images,
            labels,
            class_weight=class_weights,
            return_dict=True,
        )

        elapsed = perf_counter() - start
        timings.append(elapsed)

        assert all(
            np.isfinite(float(value)) for value in result.values()
        ), "Non-finite training result."

        print(f"Batch {step}: {elapsed:.3f} seconds", flush=True)

    average = sum(timings) / len(timings)

    print("Average seconds per batch:", round(average, 3))
    print(
        "Estimated training-only minutes per epoch:",
        round(average * len(finetune_train_dataset) / 60, 1),
    )

finally:
    # Restore every trainable weight and discard the trial optimizer state.
    for variable, original in zip(
        hybrid_model.trainable_variables, original_weights
    ):
        variable.assign(original)

    hybrid_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy")
        ],
    )

    print("Trial updates undone; optimizer reset.")

Running warm-up batch...
Batch 1: 10.011 seconds
Batch 2: 11.459 seconds
Batch 3: 10.985 seconds
Average seconds per batch: 10.819
Estimated training-only minutes per epoch: 1350.3
Trial updates undone; optimizer reset.


In [16]:
# Benchmark six batches in one continuous training call.
# Exclude the first three batches from timing and undo trial updates.

from time import perf_counter
import numpy as np
import keras

class BatchTimer(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.times = []

    def on_train_batch_begin(self, batch, logs=None):
        self.started = perf_counter()

    def on_train_batch_end(self, batch, logs=None):
        elapsed = perf_counter() - self.started
        self.times.append(elapsed)
        print(f"Batch {batch + 1}: {elapsed:.3f} seconds", flush=True)

        if not np.isfinite(float(logs["loss"])):
            raise RuntimeError("Non-finite loss; stopping the benchmark.")


original_weights = [
    variable.numpy().copy()
    for variable in hybrid_model.trainable_variables
]
timer = BatchTimer()

try:
    hybrid_model.fit(
        finetune_train_dataset.take(6),
        epochs=1,
        class_weight=class_weights,
        callbacks=[timer],
        verbose=0,
    )

    average = np.mean(timer.times[-3:])
    print("Average of last 3 batches:", round(float(average), 3))
    print(
        "Estimated training minutes per epoch:",
        round(float(average) * len(finetune_train_dataset) / 60, 1),
    )

finally:
    for variable, original in zip(
        hybrid_model.trainable_variables, original_weights
    ):
        variable.assign(original)

    hybrid_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy")
        ],
    )

    print("Trial updates undone; optimizer reset.")

Batch 1: 21.020 seconds
Batch 2: 0.574 seconds
Batch 3: 0.357 seconds
Batch 4: 0.304 seconds
Batch 5: 0.275 seconds
Batch 6: 0.411 seconds
Average of last 3 batches: 0.33
Estimated training minutes per epoch: 41.2
Trial updates undone; optimizer reset.


In [17]:
# Fine-tune for one epoch and save to a separate checkpoint.
# Preserve the frozen baseline for comparison and recovery.

checkpoint_dir.mkdir(exist_ok=True)

finetune_path = (
    checkpoint_dir / "hybrid_finetune_epoch01.weights.h5"
)

finetune_history = hybrid_model.fit(
    finetune_train_dataset,
    validation_data=finetune_validation_dataset,
    class_weight=class_weights,
    epochs=1,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            filepath=str(finetune_path),
            save_weights_only=True,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(
            str(checkpoint_dir / "hybrid_finetune_epoch01.csv")
        ),
        keras.callbacks.TerminateOnNaN(),
    ],
    verbose=1,
)

print("Fine-tuning run finished.")
print("Checkpoint path:", finetune_path)

7489/7489 ━━━━━━━━━━━━━━━━━━━━ 0s 426ms/step - accuracy: 0.7423 - loss: 0.9689
Epoch 1: saving model to checkpoints/hybrid_finetune_epoch01.weights.h5
7489/7489 ━━━━━━━━━━━━━━━━━━━━ 3538s 470ms/step - accuracy: 0.7423 - loss: 0.9689 - val_accuracy: 0.7773 - val_loss: 0.6704
Fine-tuning run finished.
Checkpoint path: checkpoints/hybrid_finetune_epoch01.weights.h5


In [18]:
# Collect validation predictions from the fine-tuned image model.
# Preserve labels in the same order and show progress.

prediction_batches = []
label_batches = []

for step, (images, labels) in enumerate(
    finetune_validation_dataset, start=1
):
    batch_predictions = hybrid_model.predict_on_batch(images)

    assert np.isfinite(batch_predictions).all()
    prediction_batches.append(batch_predictions)
    label_batches.append(labels.numpy())

    if step % 200 == 0:
        print(
            f"Processed {step}/{len(finetune_validation_dataset)} batches",
            flush=True,
        )

finetune_probabilities = np.concatenate(prediction_batches)
finetune_actual_labels = np.concatenate(label_batches)
finetune_predicted_labels = np.argmax(
    finetune_probabilities, axis=1
)

np.testing.assert_array_equal(
    finetune_actual_labels, validation_labels
)

# Save predictions so we won't need to repeat this image-processing pass.
np.savez(
    results_dir / "finetune_epoch01_validation_predictions.npz",
    filenames=np.array([r["image_id"] for r in validation_records]),
    actual_labels=finetune_actual_labels,
    predicted_labels=finetune_predicted_labels,
    probabilities=finetune_probabilities,
)

print("Fine-tuned validation predictions saved.")

Processed 200/1605 batches
Processed 400/1605 batches
Processed 600/1605 batches
Processed 800/1605 batches
Processed 1000/1605 batches
Processed 1200/1605 batches
Processed 1400/1605 batches
Processed 1600/1605 batches
Fine-tuned validation predictions saved.


In [19]:
# Calculate fine-tuned validation metrics and compare with the baseline.
# Use the saved predictions; no model inference or training is needed.

finetune_confusion = np.zeros((5, 5), dtype=np.int64)
np.add.at(
    finetune_confusion,
    (finetune_actual_labels, finetune_predicted_labels),
    1,
)

names = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
finetune_class_metrics = {}

print("Class      Precision   Recall      F1     Images")

for index, name in enumerate(names):
    correct = int(finetune_confusion[index, index])
    actual = int(finetune_confusion[index].sum())
    predicted = int(finetune_confusion[:, index].sum())

    precision = correct / predicted if predicted else 0.0
    recall = correct / actual if actual else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    finetune_class_metrics[name] = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": actual,
    }

    print(
        f"{name:<10} {precision:>8.2%}  {recall:>8.2%}  "
        f"{f1:>8.2%}  {actual:>6}"
    )

finetune_metrics = {
    "accuracy": float(np.mean(
        finetune_predicted_labels == finetune_actual_labels
    )),
    "macro_f1": float(np.mean([
        values["f1"] for values in finetune_class_metrics.values()
    ])),
    "balanced_accuracy": float(np.mean([
        values["recall"] for values in finetune_class_metrics.values()
    ])),
    "per_class": finetune_class_metrics,
    "class_order": names,
    "confusion_matrix": finetune_confusion.tolist(),
}

with (results_dir / "frozen_baseline_validation.json").open() as file:
    baseline = json.load(file)

print("\nMetric                 Baseline   Fine-tuned")
for key in ["accuracy", "macro_f1", "balanced_accuracy"]:
    print(
        f"{key:<22} {baseline[key]:>8.2%}  "
        f"{finetune_metrics[key]:>10.2%}"
    )

print(
    "\nCBSD predicted Healthy:",
    int(finetune_confusion[1, 4]),
    "(baseline: 42)",
)
print("\nConfusion matrix: rows = actual, columns = predicted")
print(finetune_confusion)

with (results_dir / "finetune_epoch01_validation.json").open("w") as file:
    json.dump(finetune_metrics, file, indent=2)

print("\nFine-tuning metrics saved.")

Class      Precision   Recall      F1     Images
CBB          43.26%    57.06%    49.21%     163
CBSD         59.13%    66.16%    62.45%     328
CGM          58.19%    77.37%    66.43%     358
CMD          95.84%    84.09%    89.58%    1974
Healthy      59.05%    64.08%    61.46%     387

Metric                 Baseline   Fine-tuned
accuracy                 79.50%      77.73%
macro_f1                 66.96%      65.83%
balanced_accuracy        69.44%      69.75%

CBSD predicted Healthy: 47 (baseline: 42)

Confusion matrix: rows = actual, columns = predicted
[[  93   29    4    7   30]
 [  30  217   22   12   47]
 [  14   16  277   27   24]
 [  25   77  141 1660   71]
 [  53   28   32   26  248]]

Fine-tuning metrics saved.


In [20]:
# Restore the better baseline and freeze its feature-extraction branches.
# Keep the fine-tuned checkpoint and results for our experiment record.

hybrid_model.load_weights(
    str(checkpoint_dir / "hybrid_frozen_baseline.weights.h5")
)

hybrid_model.get_layer("densenet121").trainable = False
hybrid_model.get_layer("swin2_tiny_256").trainable = False

print("Frozen baseline restored.")

Frozen baseline restored.


/Users/hf/Documents/CassaDx/.venv-train/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:576: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
